# Training: nested context-state encoder, centralized critic, and both actors (pursuit-evasion)
All components are trained jointly from scratch under centralized training with decentralized execution, alternating a representation step on Eq. (3) (`Train_Encoder`) with a game step on the critic and both actors (`Train_Critic`, `Train_Player1`, `Train_Player2`).

During training, the pursuer's behavior policy is the critic's equilibrium row mixed with 20% uniform exploration, plus ε-greedy exploration that decays from 1.0 to 0.05. The evader plays its network, with ε-greedy exploration that decays from 1.0 to 0.05. Episodes end at capture or after 40 rounds, so the number of environment steps varies; it is counted and saved to `main_budget.json`, which `baselines.ipynb` reads to give every baseline the same interaction budget.

In [ ]:
%run Initialize.ipynb

## Settings

In [ ]:
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

GAMMA = 0.99
CRITIC_LR = 1e-5
PLAYER1_LR = 1e-5
PLAYER2_LR = 1e-5
ENCODER_LR = 5e-5
CRITIC_BATCH = 128
PLAYER1_BATCH = 128
PLAYER2_BATCH = 128
CONTEXT_BATCH = 128
TAU = 5e-4

# Eq. (3) weights, the Soft-InfoNCE temperature, and the kernel's locality scale
LAMBDA_R = 0.1
LAMBDA_P = 0.1
LAMBDA_THETA = 0.2
SI_TEMPERATURE = 0.5
SIGMA_THETA = 1.0

a_eps, a_eps_decay, a_eps_end = 1.0, 0.9975, 0.05
u_eps, u_eps_decay, u_eps_end = 1.0, 0.997, 0.05

num_episodes = 5000
Update_Every = 10
UPDATE_START = 20

## Networks, optimizers, replay buffers

In [ ]:
# Representation: context encoder, its auxiliary heads, and EMA targets
Context_Model = TSTransformer().to(device)
trans_pred  = nn.Sequential(nn.Linear(Embedding_Size + n_a + d, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                            nn.Linear(256, Embedding_Size)).to(device)
reward_head = nn.Sequential(nn.Linear(Embedding_Size, 128), nn.ReLU(), nn.Linear(128, 1)).to(device)
proj_online = nn.Sequential(nn.Linear(Embedding_Size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
pred_online = nn.Sequential(nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
target_model = copy.deepcopy(Context_Model)
proj_target  = copy.deepcopy(proj_online)
for p_ in list(target_model.parameters()) + list(proj_target.parameters()):
    p_.requires_grad_(False)
target_model.eval(); proj_target.eval()
encoder_optimizer = optim.AdamW(list(Context_Model.parameters()) + list(trans_pred.parameters()) +
                                list(reward_head.parameters()) + list(proj_online.parameters()) +
                                list(pred_online.parameters()), lr=ENCODER_LR, amsgrad=True)

# Game: centralized critic and both actors
critic = Critic().to(device)
critic_target = Critic().to(device)
critic_target.load_state_dict(critic.state_dict())
critic_optimizer = optim.AdamW(critic.parameters(), lr=CRITIC_LR, amsgrad=True)

Player1 = DefenderActor().to(device)
player1_optimizer = optim.AdamW(Player1.parameters(), lr=PLAYER1_LR, amsgrad=True)
Player2 = AttackerActor().to(device)
player2_optimizer = optim.AdamW(Player2.parameters(), lr=PLAYER2_LR, amsgrad=True)

exp_memory = ReplayMemory(5000)
context_memory = ReplayMemory(500)
All_Loss = np.zeros([4, num_episodes])          # encoder, critic, pursuer, evader
env_steps = 0

## Training

In [ ]:
T = lambda v, dtype=torch.float32: torch.tensor(v, dtype=dtype, device=device)

for i_episode in tqdm(range(num_episodes)):

    Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col = sample_config()
    x = build_x(p_row, p_col, e_row, e_col)
    z_s = init_z_s(p_row, p_col)
    Context = np.zeros([max_steps, CTX_DIM], dtype=np.float32)
    m2 = init_m2(e_row, e_col)
    theta_Tensor, type_theta_Tensor = T(Theta), T(Type_Theta)

    for t in range(max_steps):
        mask = Context_Mask(t)
        Context_Tensor = T(Context).view(1, max_steps, CTX_DIM)
        z_s_Tensor, x_Tensor, m2_Tensor = T(z_s), T(x), T(m2)
        Context_Model.eval()
        with torch.no_grad():
            Embeding = Context_Model(Context_Tensor, mask).squeeze(0)
            Q = critic(z_s_Tensor, Embeding, x_Tensor, m2_Tensor, type_theta_Tensor, theta_Tensor).cpu().numpy()
            _, a_dist, _ = MatrixValue(Q.reshape(n_a, n_u))
            a_dist = 0.8 * a_dist + 0.2 * np.full(n_a, 1 / n_a)
            u_dist = Player2(x_Tensor, m2_Tensor, type_theta_Tensor).cpu().numpy()
        a_dist = a_dist / a_dist.sum()
        u_dist = u_dist / u_dist.sum()

        if np.random.rand() < a_eps:
            a = int(random.choices(range(n_a), weights=np.ones([n_a]) / n_a)[0])
        else:
            a = int(np.random.choice(n_a, p=a_dist))
        if np.random.rand() < u_eps:
            u = int(random.choices(range(n_u), weights=np.ones([n_u]) / n_u)[0])
        else:
            u = int(np.random.choice(n_u, p=u_dist))

        new_p_row, new_p_col, new_e_row, new_e_col = next_state(p_row, p_col, e_row, e_col, a, u, Epi_Layout)
        next_x = build_x(new_p_row, new_p_col, new_e_row, new_e_col)
        Captured = is_capture((p_row, p_col), (e_row, e_col), (new_p_row, new_p_col), (new_e_row, new_e_col))
        timeout = (t == max_steps - 1)
        done = Captured or timeout
        r = Reward(new_p_row, new_p_col, new_e_row, new_e_col, exit_rc, timeout, Captured)
        next_m2 = m2_update(m2, e_row, e_col, u, new_e_row, new_e_col)
        y = Observation(new_e_row, new_e_col)
        next_z_s = Def_State_Blf(z_s, y, new_p_row, new_p_col)
        env_steps += 1

        Context[t, :] = Context_Row(z_s, a, next_z_s)

        if t > 0:
            next_mask = Context_Mask(t + 1)
            Next_Context_Tensor = T(Context).view(1, max_steps, CTX_DIM)
            a_Tensor, u_Tensor, Reward_Tensor = T(a, torch.long), T(u, torch.long), T(r)
            exp_memory.push([x_Tensor.cpu(), T(next_x).cpu(), a_Tensor.cpu(), u_Tensor.cpu(), Reward_Tensor.cpu(),
                             T(done, torch.bool).cpu(), type_theta_Tensor.cpu(), theta_Tensor.cpu(),
                             m2_Tensor.cpu(), T(next_m2).cpu(), z_s_Tensor.cpu(), T(next_z_s).cpu(),
                             Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu()])
            context_memory.push([Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu(),
                                 a_Tensor.cpu(), T(y).cpu(), Reward_Tensor.cpu(), theta_Tensor.cpu()])

        if i_episode >= UPDATE_START and t % Update_Every == 0:
            All_Loss[0, i_episode] += Train_Encoder()          # representation step, Eq. (3)
            All_Loss[1, i_episode] += Train_Critic()           # game step
            All_Loss[2, i_episode] += Train_Player1()
            All_Loss[3, i_episode] += Train_Player2()

        if done:
            break
        x, z_s, m2 = next_x, next_z_s, next_m2
        p_row, p_col, e_row, e_col = new_p_row, new_p_col, new_e_row, new_e_col

    a_eps = max(a_eps_end, a_eps * a_eps_decay)
    u_eps = max(u_eps_end, u_eps * u_eps_decay)

## Save

In [ ]:
print(f"environment steps used in training: {env_steps:,d}")
torch.save(Context_Model.state_dict(), 'encoder.pth')
torch.save(critic.state_dict(), 'critic.pth')
torch.save(Player1.state_dict(), 'player1.pth')
torch.save(Player2.state_dict(), 'player2.pth')
with open('main_budget.json', 'w') as f:
    json.dump({'env_steps': env_steps, 'episodes': num_episodes}, f)